# Stacking with TabPFN-3

This notebook builds upon the stacking notebook from Chris: https://www.kaggle.com/code/cdeotte/gpu-logistic-regression-stacker

It replaces the stacking model with TabPFN-3 and also incorporates the additional features.

### What is TabPFN?

TabPFN (Tabular Prior-data Fitted Network) is a transformer pre-trained on millions of synthetic tabular datasets. Rather than training from scratch on your data, it performs *in-context learning*: your training rows go in as context and predictions come straight out. The recent TabPFN-3 model scales to one million rows, which is what allows us to directly tackle the dataset at hand.

| | |
|---|---|
| **Docs** | https://docs.priorlabs.ai |
| **GitHub** | https://github.com/PriorLabs/TabPFN |
| **TabPFN-3: Technical Report** | https://arxiv.org/abs/2605.13986 |
| **Hugging Face** | https://huggingface.co/Prior-Labs/tabpfn_3 |
| **Meet TabPFN-3** | https://priorlabs.ai/tabpfn |
| **Prior Labs** | https://priorlabs.ai |
| **Install** | `pip install tabpfn` |

In [1]:
!uv pip install tabpfn --system -q

In [2]:
import os
os.environ["TABPFN_MODEL_CACHE_DIR"] = "/kaggle/input/models/prior-labsai/tabpfn-3/pytorch/default/1"

In [3]:
import numpy as np
import pandas as pd
import warnings
import gc
import time
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

N_FOLDS     = 5
N_SEEDS     = 5
SEEDS       = list(range(42, 42 + N_SEEDS))
EPS         = 1e-15
LOGIT_CLIP  = 30.0
epochs      = 1000
C           = 1e-1

TARGET      = 'class'
target_map  = {'GALAXY': 0, 'QSO': 1, 'STAR': 2}
inv_map     = {v: k for k, v in target_map.items()}
BOOST       = 1.0

VER         = 1

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'Seeds: {SEEDS}')

class PyTorchMultiLogReg(nn.Module):
    def __init__(self, in_features, out_features=3):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x):
        return self.linear(x)

# Format: ('MODEL_NAME', 'oof_file.npy', 'test_file.npy')
# Update paths here if your model outputs live in another folder.
STACKING_FILES = [
    ('xgb-0', '/kaggle/input/notebooks/cdeotte/xgb-v0-for-s6e6/oof_xgb_cv.csv', '/kaggle/input/notebooks/cdeotte/xgb-v0-for-s6e6/test_xgb_preds.csv'),
    ('xgb-1', '/kaggle/input/notebooks/cdeotte/xgb-v1-for-s6e6/oof_preds.npy', '/kaggle/input/notebooks/cdeotte/xgb-v1-for-s6e6/test_preds.npy'),
    ('realmlp-0', '/kaggle/input/notebooks/yekenot/ps-s6-e6-realmlp-pytorch/oof_preds.csv', '/kaggle/input/notebooks/yekenot/ps-s6-e6-realmlp-pytorch/test_preds.csv'),
    ('realmlp-1', '/kaggle/input/notebooks/cdeotte/realmlp-v1-for-s6e6/oof_preds.npy', '/kaggle/input/notebooks/cdeotte/realmlp-v1-for-s6e6/test_preds.npy'),
    ('tabm-0', '/kaggle/input/notebooks/donmarch14/s6e6-tabm/oof_preds.csv', '/kaggle/input/notebooks/donmarch14/s6e6-tabm/test_preds.csv'),
    ('cat-0', '/kaggle/input/notebooks/cdeotte/cat-v0-for-s6e6/catboost_oof_predictions.csv', '/kaggle/input/notebooks/cdeotte/cat-v0-for-s6e6/catboost_test_predictions.csv'),
]

print(f'Total models: {len(STACKING_FILES)}')


Using device: cuda
Seeds: [42, 43, 44, 45, 46]
Total models: 6


In [4]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/train.csv', index_col='id')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/test.csv',  index_col='id')


y = train[TARGET].map(target_map).astype(int).values
N = len(y)
M = len(test)
print(f'Train: {N:,} | Test: {M:,}')
print(f'Class dist: {dict(zip(*np.unique(y, return_counts=True)))}')

def prob_to_logit(p):
    p = np.clip(p, EPS, 1.0 - EPS).astype(np.float64)
    return np.clip(np.log(p / (1.0 - p)), -LOGIT_CLIP, LOGIT_CLIP).astype(np.float32)

def load_preds(path, expected_rows=None):
    if path.lower().endswith(".csv"):
        df = pd.read_csv(path)

        # Case 1: broken flattened CSV with one column of length expected_rows * 3
        if df.shape[1] == 1:
            vals = df.iloc[:, 0].values

            if expected_rows is not None:
                assert len(vals) == expected_rows * 3, (
                    f'{path}: one-column CSV has length {len(vals)}, '
                    f'expected {expected_rows * 3}'
                )

            assert len(vals) % 3 == 0, (
                f'{path}: one-column CSV length {len(vals)} is not divisible by 3'
            )

            return vals.reshape(-1, 3)

        # Case 2: normal CSV; use last 3 columns as probabilities
        return df.iloc[:, -3:].values[:expected_rows]

    arr = np.load(path)

    if arr.ndim == 3:
        arr = arr.mean(axis=0)

    return arr

loaded_oofs  = []
loaded_tests = []
display_names = []

print('\nLoading models:')
for name, oof_f, test_f in STACKING_FILES:
    try:
        o = load_preds(oof_f, expected_rows=N)
        o = prob_to_logit(o)

        t = load_preds(test_f, expected_rows=M)
        t = prob_to_logit(t)

        assert o.shape == (N, 3), f'OOF shape {o.shape} != {(N, 3)}'
        assert t.shape == (M, 3), f'Test shape {t.shape} != {(M, 3)}'

        loaded_oofs.append(o)
        loaded_tests.append(t)
        display_names.append(name)

        print(f'  {name:20s} OOF={o.shape} TEST={t.shape}')

    except Exception as e:
        print(f'  SKIP {name} ({oof_f}, {test_f}): {e}')

n_models = len(loaded_oofs)
if n_models == 0:
    raise RuntimeError('No stacking files loaded. Update STACKING_FILES paths and rerun.')

print(f'\nModels: {n_models} | LR input features: {n_models * 3}')

X_train = np.concatenate(loaded_oofs, axis=1).astype(np.float32)

X_test = np.concatenate(loaded_tests, axis=1).astype(np.float32)

Train: 577,347 | Test: 247,435
Class dist: {np.int64(0): np.int64(377480), np.int64(1): np.int64(117143), np.int64(2): np.int64(82724)}

Loading models:
  xgb-0                OOF=(577347, 3) TEST=(247435, 3)
  xgb-1                OOF=(577347, 3) TEST=(247435, 3)
  realmlp-0            OOF=(577347, 3) TEST=(247435, 3)
  realmlp-1            OOF=(577347, 3) TEST=(247435, 3)
  tabm-0               OOF=(577347, 3) TEST=(247435, 3)
  cat-0                OOF=(577347, 3) TEST=(247435, 3)

Models: 6 | LR input features: 18


In [5]:
X_tr = train.copy()
del X_tr["class"]

# append additional features
X_train = np.concatenate([X_train, X_tr], axis=1)
X_test = np.concatenate([X_test, test], axis=1)

In [6]:
from tabpfn import TabPFNClassifier

clf = TabPFNClassifier(
    device=["cuda:0", "cuda:1"],
    n_estimators=2,
    balance_probabilities=True
)
clf.fit(X_train, train["class"])

TabPFNClassifier(balance_probabilities=True, device=['cuda:0', 'cuda:1'],
                 n_estimators=2)

In [7]:
predictions = clf.predict(X_test)

In [8]:
sub = pd.read_csv("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")
sub["class"] = predictions
sub.to_csv("submission.csv", index=False)
sub.head()

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY
